In [ ]:

from pathlib import Path

baseDir = Path("data")
modelName = "sentence-transformers/all-MiniLM-L6-v2"
minTextLength = 15
topicThreshold = 0.32
fallbackMin = 0.14
maxTopics = 6
batchSize = 128


In [6]:
topicSpecs = [
    (
        "national symbols",
        "references to nation, flags, borders, sovereignty, heritage, homeland, national history, cultural symbols",
    ),
    (
        "out-groups & boundaries",
        "references to groups described as outside or not belonging: immigrants, minorities, religious or ethnic groups, demographic threat, exclusion boundaries",
    ),
    (
        "elites & power",
        "references to elites, globalists, institutions, ngos, financial actors, international organizations, or other actors described as influential or controlling",
    ),
    (
        "street actions",
        "offline mobilization and action: marches, demonstrations, stickers, flyering, patrols, take-the-streets calls, direct action",
    ),
    (
        "ideological texts",
        "references to manifestos, theory, ideological texts, historical revisionism, canonical authors, internal debates",
    ),
    (
        "electoral politics",
        "elections and party politics: parties, candidates, voting, campaigns, parliament, coalitions",
    ),
    (
        "media & information",
        "references to media outlets, journalists, platforms, censorship, bans, channels, or information control",
    ),
    (
        "security & violence",
        "references to police, military, weapons, militias, conflict, violence, or threats of force",
    ),
    (
        "territory & space",
        "references to cities, regions, borders, territory, land, invasion, defense, or spatial control",
    ),
]

topicLabels = [label for label, _ in topicSpecs]
topicLabelTexts = [f"{label} — {desc}" for label, desc in topicSpecs]


In [7]:

import re

URL_PATTERN = re.compile(r"https?://\S+")
RT_PATTERN = re.compile(r"^rt\s+@\w+:\s*", re.IGNORECASE)
MENTION_PATTERN = re.compile(r"@\w+")
MULTISPACE_PATTERN = re.compile(r"\s+")
CONTROL_CHARS_PATTERN = re.compile(r"[\u0000-\u001f\u007f-\u009f]")

def canonicalizeKey(text: str) -> str:
    text = text or ""
    text = URL_PATTERN.sub(" ", text)
    text = RT_PATTERN.sub("", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = CONTROL_CHARS_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text.lower()

def cleanForInference(text: str) -> str:
    text = text or ""
    text = URL_PATTERN.sub(" ", text)
    text = RT_PATTERN.sub("", text)
    text = CONTROL_CHARS_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text


In [8]:

import torch
from sentence_transformers import SentenceTransformer

def chooseDevice() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

device = chooseDevice()
topicModel = SentenceTransformer(modelName, device=device)
topicEmbeddings = topicModel.encode(
    topicLabelTexts,
    convert_to_tensor=True,
    normalize_embeddings=True,
)


In [9]:

import numpy as np
from sentence_transformers import util

def predictTopicsBatch(texts):
    cleaned = [cleanForInference(t) for t in texts]
    embeddings = topicModel.encode(
        cleaned,
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=batchSize,
        show_progress_bar=False,
    )
    scores = util.dot_score(embeddings, topicEmbeddings).cpu().numpy()
    out = []
    for row in scores:
        order = np.argsort(-row)
        best_score = float(row[order[0]]) if len(order) else -1
        selected = [topicLabels[i] for i in order if float(row[i]) >= topicThreshold]
        if not selected and best_score >= fallbackMin and len(order):
            selected = [topicLabels[int(order[0])]]
        out.append(selected[:maxTopics])
    return out


In [10]:

import pandas as pd
sampleSize = 2000

def build_sample() -> pd.DataFrame:
    chunks = []
    seen = set()
    for path in sorted(baseDir.glob("*/message_nodes.csv")):
        df = pd.read_csv(path)
        if "text" not in df.columns:
            continue
        df = df[["text"]].copy()
        df["text"] = df["text"].fillna("").astype(str)
        df = df[df["text"].str.len() >= minTextLength]
        if df.empty:
            continue
        df["normText"] = df["text"].apply(canonicalizeKey)
        df = df[df["normText"].map(bool)]
        df = df[~df["normText"].isin(seen)]
        if df.empty:
            continue
        seen.update(df["normText"].tolist())
        df["dataset"] = path.parent.name
        df = df.head(sampleSize)
        chunks.append(df)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=["text", "normText", "dataset"])

sample = build_sample()
if sample.empty:
    print("No sample data available to inspect")
else:
    texts = sample["text"].tolist()
    sample_topics = predictTopicsBatch(texts)
    counts = pd.Series([tuple(t) for t in sample_topics]).value_counts()
    print("Sample topics (top 5):")
    print(counts.head(5))


Sample topics (top 5):
(ideological texts,)      2000
(street actions,)         1188
(national symbols,)       1009
(media & information,)     878
(security & violence,)     836
Name: count, dtype: int64


In [11]:

import pandas as pd
from tqdm.auto import tqdm

messagePaths = sorted(baseDir.glob("*/message_nodes.csv"))
for path in tqdm(messagePaths, desc="tagging message_nodes", unit="file"):
    df = pd.read_csv(path)
    if "text" not in df.columns:
        continue
    df["text"] = df["text"].fillna("").astype(str)
    df["normText"] = df["text"].apply(canonicalizeKey)
    unique = df.drop_duplicates(subset="normText")
    valid = unique[
        (unique["text"].str.len() >= minTextLength) & unique["normText"].astype(bool)
    ].copy()
    topicMap = {}
    if not valid.empty:
        keys = valid["normText"].tolist()
        texts = valid["text"].tolist()
        preds = []
        for i in range(0, len(texts), batchSize):
            preds.extend(predictTopicsBatch(texts[i : i + batchSize]))
        topicMap = dict(zip(keys, preds))
    df["topics"] = df["normText"].map(lambda key: topicMap.get(key, []) if key else [])
    df = df.drop(columns=["normText"])
    df.to_csv(path, index=False)


tagging message_nodes:   0%|          | 0/4 [00:00<?, ?file/s]

In [12]:

import json
from tqdm.auto import tqdm

graphPaths = sorted(baseDir.glob("*/graph.json"))
for path in tqdm(graphPaths, desc="tagging graphs", unit="file"):
    graph = json.loads(path.read_text(encoding="utf-8"))
    messages = graph.get("messages", [])
    if not messages:
        continue
    texts = [(m.get("text") or "").strip() for m in messages]
    keys = [canonicalizeKey(t) for t in texts]
    unique = {}
    for key, text in zip(keys, texts):
        if not key or len(text) < minTextLength or key in unique:
            continue
        unique[key] = text
    topicMap = {}
    if unique:
        uniq_keys = list(unique.keys())
        uniq_texts = [unique[k] for k in uniq_keys]
        preds = []
        for i in range(0, len(uniq_texts), batchSize):
            batch = uniq_texts[i : i + batchSize]
            preds.extend(predictTopicsBatch(batch))
        topicMap = dict(zip(uniq_keys, preds))
    for msg, key, text in zip(messages, keys, texts):
        msg["topics"] = topicMap.get(key, []) if key and len(text) >= minTextLength else []
    path.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding="utf-8")


tagging graphs:   0%|          | 0/4 [00:00<?, ?file/s]

In [13]:

import shutil

staticRoot = Path("../app/static/data")
for datasetDir in sorted(baseDir.iterdir()):
    if not datasetDir.is_dir():
        continue
    targetDir = staticRoot / datasetDir.name
    targetDir.mkdir(parents=True, exist_ok=True)
    for filename in ("message_nodes.csv", "graph.json"):
        src = datasetDir / filename
        if not src.exists():
            continue
        dest = targetDir / filename
        shutil.copy2(src, dest)
        print(f"Copied {src} → {dest}")


Copied data/afdjugendbw/message_nodes.csv → ../app/static/data/afdjugendbw/message_nodes.csv
Copied data/afdjugendbw/graph.json → ../app/static/data/afdjugendbw/graph.json
Copied data/generationidentitaire/message_nodes.csv → ../app/static/data/generationidentitaire/message_nodes.csv
Copied data/generationidentitaire/graph.json → ../app/static/data/generationidentitaire/graph.json
Copied data/jungenationalisten/message_nodes.csv → ../app/static/data/jungenationalisten/message_nodes.csv
Copied data/jungenationalisten/graph.json → ../app/static/data/jungenationalisten/graph.json
Copied data/tricoloredelsangueitalico/message_nodes.csv → ../app/static/data/tricoloredelsangueitalico/message_nodes.csv
Copied data/tricoloredelsangueitalico/graph.json → ../app/static/data/tricoloredelsangueitalico/graph.json
